# 03 — Function objects and composition

**Objectives.** Create named multi-input functions, call them numerically and symbolically, compose graphs, and use mapping. A `Function` is the main interface shared by integrators, root finders, and solvers.


In [10]:
# Make repository-local helpers importable without installing a package.
import sys
from pathlib import Path

repository_root = next(
    candidate
    for candidate in (Path.cwd(), *Path.cwd().parents)
    if (candidate / "casadi_tutorial").is_dir()
)
if str(repository_root) not in sys.path:
    sys.path.insert(0, str(repository_root))

# Autoreload picks up edits to tutorial models before each notebook cell runs.
ipython = get_ipython()
if ipython is not None:
    ipython.run_line_magic("load_ext", "autoreload")
    ipython.run_line_magic("autoreload", "2")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


### Build a multi-output function and inspect its graph

This cell defines a weighted quadratic energy with SX symbols. `ca.dot`
forms the inner product, and `ca.gradient` differentiates the scalar
energy with respect to the complete state vector. `ca.Function` packages
the value and gradient in one reusable graph with named inputs and
outputs. A subsequent symbolic call is concatenated and passed to
`dotdraw`, which asks Graphviz to render the dependency graph.

For $x\in\mathbb{R}^2$ and scalar $p$,

$$
E(x,p)=p\,x^\mathsf{T}x
      =p(x_0^2+x_1^2),
\qquad
\nabla_xE(x,p)=2px.
$$

Returning both quantities from one function allows CasADi to share common
subexpressions and gives callers a single consistent interface.


In [ ]:
import casadi as ca
from casadi.tools.graph import dotdraw

x = ca.SX.sym("x", 2)
weight = ca.SX.sym("weight")

# dot forms an inner product, and gradient differentiates a scalar with
# respect to every element of the selected symbolic state x.
energy = weight * ca.dot(x, x)
energy_gradient = ca.gradient(energy, x)

# Named inputs/outputs make calls and generated interfaces self-documenting.
energy_function = ca.Function(
    "energy",
    [x, weight],
    [energy, energy_gradient],
    ["x", "weight"],
    ["value", "gradient"],
)
print(energy_function)
print(energy_function(x=[1, 2], weight=0.5))

# A symbolic Function call is still a graph, so dotdraw can inspect it.
symbolic_value, symbolic_gradient = energy_function(x, weight)
dotdraw(
    ca.vertcat(symbolic_value, symbolic_gradient),
    direction="LR",
)

### Map repeated evaluations and compare expanded graphs

`Function.map(3)` creates three independent copies of the energy function
in one CasADi call. Each column of the input matrices is one sample, so no
Python loop is needed. The cell then nests a function call inside a new
graph and uses `expand()` to replace higher-level MX-style calls with
scalar SX operations. `n_nodes()` compares the graph sizes before and
after that transformation.

For samples $x^{(j)}$ and weights $p^{(j)}$, mapping evaluates

$$
\left\{E\!\left(x^{(j)},p^{(j)}\right),
\nabla_xE\!\left(x^{(j)},p^{(j)}\right)\right\}_{j=0}^{2}.
$$

The nested scalar output is

$$
g(x)=E(x,2)+\sin(x_0).
$$

Expansion is a representation choice rather than a mathematical change:
the expanded and unexpanded functions compute the same $g(x)$.


In [ ]:
x_samples = ca.DM([[1, 2, 3], [4, 5, 6]])

# map creates three independent copies of one Function without a Python loop.
mapped_energy = energy_function.map(3)
mapped_values, mapped_gradients = mapped_energy(
    x_samples,
    ca.DM.ones(1, 3),
)
assert mapped_values.shape == (1, 3)
assert mapped_gradients.shape == x_samples.shape

# expand converts an MX-style call graph into scalar SX operations. This can
# improve some workloads but can also increase graph size substantially.
nested_function = ca.Function(
    "nested",
    [x],
    [energy_function(x, 2)[0] + ca.sin(x[0])],
)
expanded_function = nested_function.expand()
print(
    "nodes before/after expansion:",
    nested_function.n_nodes(),
    expanded_function.n_nodes(),
)

`map(N)` expresses independent repeated evaluations without manually unrolling Python. `mapaccum` threads an output state into the next evaluation. Expansion converts eligible MX operations to an SX-style scalar graph: sometimes faster to evaluate, often larger to store.

**Common mistakes:** duplicate function names in generated code, mismatched sparsity, and assuming named calls return a tuple (they return a dictionary). **Recap:** Functions provide stable graph boundaries and introspection. [Exercises](../exercises/02_functions_ad.ipynb).
